In [1]:
# import custom modules
import sys 
sys.path.append('../src')

from data_processing import (
    load_ai_detection_dataset,
    convert_labels_to_numeric,
    apply_text_cleaning,
    create_stratified_sample,
    split_train_val_test,
    save_train_test_splits,
)

from feature_engineering import (
    create_tfidf_features,
    save_tfidf_features
)


# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path

print("Modules imported successfully")

Modules imported successfully


## Step 1: Load Dataset

Using our custom `data_processing` module to load the AI Text Detection Pile from Hugging Face. The module handles dataset loading and provides initial statistics about class distribution.

In [2]:
# Load AI text detection pile dataset
df = load_ai_detection_dataset()

# Quick quality checks
print(f"\nMissing values:\n{df.isnull().sum()}")

Loading dataset: artem9k/ai-text-detection-pile...
Dataset loaded: 1,392,522 samples

Missing values:
source    0
id        0
text      0
dtype: int64


## Step 2: Convert Labels to Numeric

Convert categorical labels ('human', 'ai') to numeric format for machine learning:
- 0 = Human-written text
- 1 = AI-generated text

In [3]:
# Convert to numeric: 1 = AI, 0 = Human
df = convert_labels_to_numeric(df)


Class Distribution:
label
0    1028146
1     364376
Name: count, dtype: int64
Ratio (Human:AI) = 2.82:1


## Step 3: Text Cleaning

Clean text data to prepare for feature engineering:
- Remove URLs (http/https/www)
- Remove extra whitespace
- Convert to lowercase
- Keep punctuation (useful for stylistics features later)

In [4]:
# Clean text data: remove URLs, normalize whitespace, lowercase
df = apply_text_cleaning(df)

# Show example of cleaning
print("\nCleaning Example")
print(f"Original: {df.loc[0, 'text'][:100]}...")
print(f"Cleaned: {df.loc[0, 'cleaned_text'][:100]}...")

Cleaning text in column 'text...
Text cleaning complete: 1392522 samples 

Cleaning Example
Original: 12 Years a Slave: An Analysis of the Film Essay

The 2013 film 12 Years a Slave proved that slavery ...
Cleaned: 12 years a slave: an analysis of the film essay the 2013 film 12 years a slave proved that slavery i...


## Step 4: Create Sample and Split Data

**Why 30% sample**
Working with the full 1.4M dataset requires significant RAM and disk space. A 30% stratified sample (417k samples) maintains the class distribution while being computationally feasible on Mac Hardware.

**Split Strategy: 70-15-15**
- **Training (70%):** Used to train models
- **Validation (15%):** Used for hyperparamter tuning and model selection
- **Test (15%):** Final evaluation, never seen during training

All splits use stratification to maintain class imbalance

In [5]:
# Create 30% stratified sample (due to Mac hardware constraints)
df_medium = create_stratified_sample(df, frac=0.3, random_state=42)

# Extract features (X) and labels (y)
X = df_medium['cleaned_text']
y = df_medium['label']

# Split into train (70%), validation (15%), test (15%) sets
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, test_size=0.15, val_size=0.15, random_state=42
)


Creating 30% stratified sample...
Sample size: 417,757
Class distribution:
label
0    308536
1    109221
Name: count, dtype: int64

Data Split Summary:
Training: 292,429 (70.0%)
Validation: 62,664 (15.0%)
Test: 62,664 (15.0%)


## Step 5: Feature Engineering - TF-IDF

**TF-IDF (Term Frequency-Inverse Document Frequency)** converts text to numerical features:

**Parameter Rationale:**
- `max_features=5000`: Captures distinctive vocabulary without overfitting
- `ngram_range=(1,2)`: Includes both single words and two-word phrases
- `min_df=5`: Filters out rare words (noise reduction)
- `max_df=0.8`: Filters out common words (e.g., "the", "and")

**Results:** 5000 numerical features per text sample, representing word importance

In [6]:
# Create TF-IDF features
X_train_tfidf, X_val_tfidf, X_test_tfidf, tfidf = create_tfidf_features(
    X_train, X_val, X_test,
    max_features=5000, # Top 5000 most important features
    ngram_range=(1, 2), # Unigrams + bigrams
    min_df=5, # Must appear in at least 5 docs
    max_df=0.8 # Ignore terms in >80% of documents
)

# Inspect some feature names
feature_names = tfidf.get_feature_names_out()
print(f"\nSample features (first 10):")
for i, feature in enumerate(feature_names[:10], 1):
    print(f"{i}. '{feature}'")


Creating TF-IDF features...
Parameters: max_features=5000, ngram_range=(1, 2)
            min_df=5, max_df=0.8
TF-IDF shape: (292429, 5000)
Vocabulary size: 5000

Sample features (first 10):
1. '00'
2. '000'
3. '10'
4. '100'
5. '11'
6. '12'
7. '13'
8. '14'
9. '15'
10. '16'


## Step 6: Save Preprocessed Data

Save both the text splits and TF-IDF features for use in model training:

1. **Text splits** (`train_test_splits_medium.pkl`): Original cleaned text
2. **TF-IDF features** (`tfidf_features_medium.pkkl`): Numerical feature matrices + fitted vectorizer

This separation allows:
- Reporducibility: Can recreate features from text
- Efficiency: Can load pre-computed features directly for training
- Flexibility: Can experiment with different feature engineering approaches

In [7]:
# Save the text splits (for reference and future use)
save_train_test_splits(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    filepath='../data/processed/train_test_splits_medium.pkl'
)

# Save TF-IDF features (these are used for training)
save_tfidf_features(
    X_train_tfidf, X_val_tfidf, X_test_tfidf,
    tfidf,
    filepath='../data/features/tfidf_features_medium.pkl'
)

print("\nAll preprocessing complete!")
print("Ready for model training in notebok 03")


Splits saved to: ../data/processed/train_test_splits_medium.pkl

TF-IDF features saved to: ../data/features/tfidf_features_medium.pkl

All preprocessing complete!
Ready for model training in notebok 03


## Preprocessing Complete

This notebook has successfully:
1. Loaded 1.4M text samples from Hugging Face
2. Converted labels to numeric format
3. Cleaned text (removed URLs, normalized formatting)
4. Created 30% stratified sample (417k samples)
5. Split into train/val/test (70-15-15)
6. Generated TF-IDF features (5000 dimensions)
7. Saved all data for model training

**Key Achievement:** Professional, modular preprocessing pipeline using custom Python modules.

In [8]:
# Summary of preprocessing pipeline
print("="*60)
print("PREPROCESSING PIPELINE SUMMARY")
print("="*60)
print(f"\nDataset: AI Text Detection Pile")
print(f"Total samples: 1,392,522")
print(f"Working Sample (30%): {len(df_medium):,}")
print(f"\nData Splits:")
print(f"Training: {len(X_train):,} samples ({len(X_train)/len(df_medium)*100:.1f}%)")
print(f"Validation: {len(X_val):,} samples ({len(X_val)/len(df_medium)*100:.1f}%)")
print(f"Test: {len(X_test):,} samples ({len(X_test)/len(df_medium)*100:.1f}%)")
print(f"Features:")
print(f"TF-IDF matrix: {X_train_tfidf.shape[0]:,} samples x {X_train_tfidf.shape[1]:,} features")
print(f"Feature type: TF-IDF with unigrams + bigrams")
print(f"Saved Files: ")
print(f"../data/processed/train_test_splits_medium.pkl")
print(f"../data/features/tfidf_features_medium.pkl")
print("="*60)

PREPROCESSING PIPELINE SUMMARY

Dataset: AI Text Detection Pile
Total samples: 1,392,522
Working Sample (30%): 417,757

Data Splits:
Training: 292,429 samples (70.0%)
Validation: 62,664 samples (15.0%)
Test: 62,664 samples (15.0%)
Features:
TF-IDF matrix: 292,429 samples x 5,000 features
Feature type: TF-IDF with unigrams + bigrams
Saved Files: 
../data/processed/train_test_splits_medium.pkl
../data/features/tfidf_features_medium.pkl
